# PE eosfit / traditional plotting notebook

This version uses **bilby's own plotting utilities** (`bilby.core.result.plot_multiple`) for the main corner plots, to stay closer to the original notebook style.


In [1]:
# ==============================
# Configuration
# ==============================
POP_OUTDIR = "/scratch/gpfs/ANDREASB/fy6204/GW/Workspace/outputs/outdir_population_exactfd"

EOSFIT_OUTDIR = "/scratch/gpfs/ANDREASB/fy6204/GW/Workspace/outputs/outdir_population_run"
TRAD_OUTDIR   = "/scratch/gpfs/ANDREASB/fy6204/GW/Workspace/outputs/outdir_population_run_traditional"

EVENT_INDEX = 5
EVENT_NAME = f"event_{EVENT_INDEX:04d}"

# Fill these with your actual labels
EOSFIT_LABEL = f"bns_{EVENT_NAME}_eosfit_dyn"
TRAD_LABEL   = f"bns_{EVENT_NAME}_traditional_Mc0.1"

# Reweighted labels in the current scripts default to label + "_reweighted"
EOSFIT_RW_LABEL = EOSFIT_LABEL + "_reweighted"
TRAD_RW_LABEL   = TRAD_LABEL + "_reweighted"

USE_REWEIGHTED_EOSFIT = False
USE_REWEIGHTED_TRAD   = False

# Prior sample size. None -> use the posterior sample count.
N_PRIOR = None
PRIOR_RANDOM_SEED = 123456

# Reweighted posteriors can contain many duplicated rows from weighted resampling.
# To reduce artificial tiny spikes in corner plots, cap repeated identical points.
CAP_REWEIGHTED_DUPLICATES_FOR_PLOT = False
MAX_DUPLICATES_PER_POINT = 3
DUPLICATE_ROUND_DECIMALS = 9
DUPLICATE_CAP_RANDOM_SEED = 24680

# Parameters for the EOS-fit / UR parameterization plot
EOSFIT_UR_PARAMS = [
    "chirp_mass",
    "mass_ratio",
    "luminosity_distance",
    "redshift_sample",
    "H0_sample",
    "delta_a0",
    "delta_a1",
    "delta_a2",
]

# Parameters common to EOS-fit and traditional, for direct physical comparison
COMMON_PHYSICAL_PARAMS = [
    "chirp_mass",
    "mass_ratio",
    "lambda_tilde",
    "delta_lambda_tilde",
    "luminosity_distance",
    "mass_1",
    "mass_2",
]


In [6]:
result=bilby.core.result.read_in_result("outputs/outdir_population_run/bns_event_0005_eosfit_dyn_result.hdf5")

16:21 bilby INFO    : Global meta data was removed from the result object for compatibility. Use the `BILBY_INCLUDE_GLOBAL_METADATA` environment variable to include it. This behaviour will be removed in a future release. For more details see: https://bilby-dev.github.io/bilby/faq.html#global-meta-data


In [7]:
result.injection_parameters

{'H0_sample': np.float64(67.66),
 'a_1': np.float64(0.0),
 'a_2': np.float64(0.0),
 'azimuth': np.float64(0.44981115793632476),
 'chi_1': np.float64(0.0),
 'chi_2': np.float64(0.0),
 'chirp_mass': np.float64(1.3326542081511827),
 'cos_tilt_1': np.float64(0.0),
 'cos_tilt_2': np.float64(0.0),
 'delta_a0': np.float64(0.0),
 'delta_a1': np.float64(0.0),
 'delta_a2': np.float64(0.0),
 'delta_lambda_tilde': np.float64(73.07788135418448),
 'geocent_time': np.float64(1126259642.413),
 'lambda_1': np.float64(525.9305737127601),
 'lambda_2': np.float64(1175.0017453165115),
 'lambda_tilde': np.float64(790.8167075354212),
 'luminosity_distance': np.float64(875.4431971941967),
 'mass_1': np.float64(1.6403998108525308),
 'mass_1_source': np.float64(1.3955658004370788),
 'mass_2': np.float64(1.4299026383993596),
 'mass_2_source': np.float64(1.2164858877103877),
 'mass_ratio': np.float64(0.8716793484975021),
 'phase': np.float64(0.20452975555898406),
 'phi_12': np.float64(0.0),
 'phi_jl': np.float64(

In [2]:
import os
import json
import copy
import warnings

import bilby
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM, Planck18

warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# Helper classes copied from the current PE scripts
# =========================================================
class UniformReflected(bilby.core.prior.analytical.Uniform):
    def rescale(self, val):
        u = 2 * np.minimum(val, 1 - val)
        return super().rescale(u)


# =========================================================
# Generic helpers
# =========================================================
def ensure_dataframe(x):
    if isinstance(x, pd.DataFrame):
        return x.copy()
    if isinstance(x, dict):
        return pd.DataFrame(x)
    return pd.DataFrame(x)


def rename_legacy_mass_columns(df):
    df = ensure_dataframe(df)
    rename_map = {}
    if "m_1" in df.columns and "mass_1" not in df.columns:
        rename_map["m_1"] = "mass_1"
    if "m_2" in df.columns and "mass_2" not in df.columns:
        rename_map["m_2"] = "mass_2"
    if rename_map:
        df = df.rename(columns=rename_map)
    return df


def load_meta(pop_outdir, event_name):
    meta_path = os.path.join(pop_outdir, event_name, "meta.json")
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return meta


def pick_existing(paths):
    for path in paths:
        if path and os.path.exists(path):
            return path
    return None


def load_base_result(outdir, label):
    candidates = [
        os.path.join(outdir, f"{label}_result.hdf5"),
        os.path.join(outdir, f"{label}_result.h5"),
        os.path.join(outdir, f"{label}_result.json"),
        os.path.join(outdir, f"{label}_result.pkl"),
    ]
    existing = [p for p in candidates if os.path.exists(p)]
    if not existing:
        raise FileNotFoundError("Cannot find result file. Tried:\n  " + "\n  ".join(candidates))

    last_err = None
    for result_path in existing:
        try:
            result = bilby.core.result.read_in_result(filename=result_path)
            return result, result_path
        except Exception as e:
            last_err = e
            continue

    raise RuntimeError(
        "Found result files but none could be read by bilby. Tried:\n  "
        + "\n  ".join(existing)
        + f"\nLast error: {last_err}"
    )


def load_posterior_dataset(outdir, label, rw_label=None, use_reweighted=False):
    base_result, base_result_path = load_base_result(outdir, label)

    summary_path = None
    summary_data = None
    if use_reweighted:
        rw_label = rw_label or (label + "_reweighted")
        csv_path = os.path.join(outdir, f"{rw_label}_posterior_augmented.csv")
        rw_result_paths = [
            os.path.join(outdir, f"{rw_label}_result.hdf5"),
            os.path.join(outdir, f"{rw_label}_result.h5"),
            os.path.join(outdir, f"{rw_label}_result.json"),
            os.path.join(outdir, f"{rw_label}_result.pkl"),
        ]

        if os.path.exists(csv_path):
            src = csv_path
            posterior_df = pd.read_csv(src)
        else:
            existing_rw = [p for p in rw_result_paths if os.path.exists(p)]
            if not existing_rw:
                raise FileNotFoundError(
                    "Cannot find a reweighted posterior source. Tried:\n  "
                    + "\n  ".join([csv_path] + rw_result_paths)
                )

            src = None
            posterior_df = None
            last_err = None
            for rp in existing_rw:
                try:
                    rw_result = bilby.core.result.read_in_result(filename=rp)
                    posterior_df = ensure_dataframe(rw_result.posterior)
                    src = rp
                    break
                except Exception as e:
                    last_err = e
                    continue

            if posterior_df is None:
                raise RuntimeError(
                    "Found reweighted result files but none could be read by bilby. Tried:\n  "
                    + "\n  ".join(existing_rw)
                    + f"\nLast error: {last_err}"
                )

        active_label = rw_label

        summary_path = os.path.join(outdir, f"{rw_label}_summary.json")
        if os.path.exists(summary_path):
            with open(summary_path, "r", encoding="utf-8") as f:
                summary_data = json.load(f)
    else:
        csv_path = os.path.join(outdir, f"{label}_posterior_augmented.csv")
        if os.path.exists(csv_path):
            src = csv_path
            posterior_df = pd.read_csv(src)
        else:
            src = base_result_path
            posterior_df = ensure_dataframe(base_result.posterior)
        active_label = label

    posterior_df = rename_legacy_mass_columns(posterior_df)
    return {
        "base_result": base_result,
        "base_result_path": base_result_path,
        "posterior_df_raw": posterior_df,
        "posterior_source": src,
        "active_label": active_label,
        "summary_path": summary_path,
        "summary_data": summary_data,
    }


def infer_posterior_sample_count(posterior):
    if isinstance(posterior, pd.DataFrame):
        return len(posterior)

    if isinstance(posterior, dict):
        # Bilby JSON-style encoded DataFrame
        if posterior.get("__dataframe__") and "content" in posterior:
            content = posterior["content"]
            if isinstance(content, dict) and len(content) > 0:
                first = next(iter(content.values()))
                if isinstance(first, dict) and first.get("__series__") and "content" in first:
                    return len(first["content"])
                if hasattr(first, "__len__"):
                    return len(first)

        # Dict of columns -> arrays/Series (or bilby JSON-style Series)
        for key, value in posterior.items():
            if str(key).startswith("__"):
                continue
            if isinstance(value, dict) and value.get("__series__") and "content" in value:
                return len(value["content"])
            if hasattr(value, "__len__"):
                try:
                    return len(value)
                except Exception:
                    pass

    if hasattr(posterior, "shape"):
        shape = getattr(posterior, "shape")
        if isinstance(shape, tuple) and len(shape) >= 1 and shape[0] is not None:
            return int(shape[0])

    return None


def sample_prior_dataframe(result, size=None, seed=123456):
    if size is None:
        size = infer_posterior_sample_count(result.posterior)
    if size is None and hasattr(result, "samples") and result.samples is not None:
        size = len(result.samples)
    if size is None:
        raise ValueError("Could not infer sample size from result.posterior; set size explicitly.")

    size = int(size)
    rng = np.random.default_rng(seed)
    np.random.seed(int(rng.integers(0, 2**31 - 1)))

    if hasattr(result.priors, "non_fixed_keys"):
        sample_keys = [k for k in result.priors.non_fixed_keys if k in result.priors]
    else:
        sample_keys = list(result.priors.keys())

    if hasattr(result.priors, "sample_subset_constrained_as_array"):
        arr = result.priors.sample_subset_constrained_as_array(
            keys=sample_keys,
            size=size,
        )
        arr = np.asarray(arr)

        if arr.shape == (len(sample_keys), size):
            arr = arr.T
        elif arr.shape != (size, len(sample_keys)):
            raise ValueError(
                f"Unexpected prior sample array shape: {arr.shape}, expected "
                f"{(len(sample_keys), size)} or {(size, len(sample_keys))}"
            )

        df = pd.DataFrame(arr, columns=sample_keys)
    else:
        samples = result.priors.sample_subset_constrained(keys=sample_keys, size=size)
        df = ensure_dataframe(samples)

    return rename_legacy_mass_columns(df.reset_index(drop=True))


def apply_result_constraints(df, result):
    """Apply hard constraint keys from bilby Result.priors to a DataFrame."""
    df = ensure_dataframe(df)
    priors = getattr(result, "priors", None)
    if priors is None:
        return df.reset_index(drop=True)

    ckeys = list(getattr(result, "constraint_parameter_keys", []) or [])
    if not ckeys:
        ckeys = [
            k for k, v in priors.items()
            if getattr(v, "__class__", type(v)).__name__ == "Constraint"
        ]

    out = df.copy()
    for k in ckeys:
        if k not in out.columns or k not in priors:
            continue
        prior_k = priors[k]
        arr = pd.to_numeric(out[k], errors="coerce")
        out = out.loc[np.isfinite(arr)]
        arr = pd.to_numeric(out[k], errors="coerce")

        vmin = getattr(prior_k, "minimum", None)
        vmax = getattr(prior_k, "maximum", None)
        if vmin is not None:
            out = out.loc[arr >= float(vmin)]
            arr = pd.to_numeric(out[k], errors="coerce")
        if vmax is not None:
            out = out.loc[arr <= float(vmax)]

    return out.reset_index(drop=True)


def finite_clean(df, params):
    df = rename_legacy_mass_columns(df)
    keep = [p for p in params if p in df.columns]
    out = df[keep].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return out


def count_duplicate_rows(df, keys, round_decimals=9):
    df = ensure_dataframe(df)
    if len(df) == 0:
        return 0, 0, 0

    key_df = df[keys].copy()
    for k in keys:
        key_df[k] = pd.to_numeric(key_df[k], errors="coerce")
    key_df = key_df.replace([np.inf, -np.inf], np.nan).dropna().round(round_decimals)
    if len(key_df) == 0:
        return len(df), 0, 0

    grp = pd.util.hash_pandas_object(key_df, index=False)
    counts = grp.value_counts()
    unique_rows = int(len(counts))
    dup_extra = int((counts - 1).clip(lower=0).sum())
    max_mult = int(counts.max())
    return unique_rows, dup_extra, max_mult


def cap_duplicate_rows_for_plot(df, keys, max_per_point=3, round_decimals=9, random_state=24680):
    """Cap repeated rows in selected keys to reduce spiky artifacts from weighted resampling."""
    df = ensure_dataframe(df)
    if len(df) == 0 or max_per_point is None:
        return df.reset_index(drop=True), {"n_in": len(df), "n_out": len(df), "changed": False}

    key_df = df[keys].copy()
    for k in keys:
        key_df[k] = pd.to_numeric(key_df[k], errors="coerce")
    key_df = key_df.replace([np.inf, -np.inf], np.nan)

    valid_mask = key_df.notna().all(axis=1)
    df_valid = df.loc[valid_mask].copy()
    key_valid = key_df.loc[valid_mask].round(round_decimals)

    if len(df_valid) == 0:
        return df.iloc[0:0].copy(), {"n_in": len(df), "n_out": 0, "changed": len(df) > 0}

    group_id = pd.util.hash_pandas_object(key_valid, index=False).to_numpy()
    df_valid["__dup_group__"] = group_id

    parts = []
    rng = np.random.default_rng(random_state)
    for _, g in df_valid.groupby("__dup_group__", sort=False):
        if len(g) <= max_per_point:
            parts.append(g)
        else:
            rs = int(rng.integers(0, 2**31 - 1))
            parts.append(g.sample(n=max_per_point, random_state=rs, replace=False))

    out = pd.concat(parts, ignore_index=True)
    out = out.drop(columns=["__dup_group__"], errors="ignore")
    out = out.reset_index(drop=True)

    return out, {
        "n_in": int(len(df)),
        "n_valid": int(len(df_valid)),
        "n_out": int(len(out)),
        "changed": bool(len(out) != len(df)),
    }


def maybe_cap_reweighted_duplicates(df, keys, enabled, max_per_point, round_decimals, random_state, tag=""):
    uniq_before, dup_before, max_mult_before = count_duplicate_rows(df, keys, round_decimals=round_decimals)
    if not enabled:
        print(f"[{tag}] duplicate cap OFF | n={len(df)}, unique={uniq_before}, dup_extra={dup_before}, max_mult={max_mult_before}")
        return df.reset_index(drop=True)

    out, info = cap_duplicate_rows_for_plot(
        df,
        keys=keys,
        max_per_point=max_per_point,
        round_decimals=round_decimals,
        random_state=random_state,
    )
    uniq_after, dup_after, max_mult_after = count_duplicate_rows(out, keys, round_decimals=round_decimals)
    print(
        f"[{tag}] duplicate cap ON | n: {len(df)} -> {len(out)}, "
        f"unique: {uniq_before} -> {uniq_after}, "
        f"dup_extra: {dup_before} -> {dup_after}, "
        f"max_mult: {max_mult_before} -> {max_mult_after}"
    )
    return out


def build_priors_for_plot(base_result, df, params):
    priors_for_plot = {}
    for p in params:
        if p in base_result.priors:
            priors_for_plot[p] = base_result.priors[p]
        else:
            x = pd.to_numeric(df[p], errors="coerce").to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            peak = float(np.nanmedian(x)) if x.size else 0.0
            priors_for_plot[p] = bilby.core.prior.DeltaFunction(peak=peak, name=p)
    return priors_for_plot


def make_lightweight_result(base_result, df, params, label_map, result_label, sampler=None, priors_for_plot=None):
    labels = [label_map.get(p, p) for p in params]
    if priors_for_plot is None:
        priors_for_plot = build_priors_for_plot(base_result, df, params)

    res = bilby.core.result.Result(
        label=result_label,
        outdir=base_result.outdir,
        sampler=base_result.sampler if sampler is None else sampler,
        search_parameter_keys=params,
        priors=priors_for_plot,
        posterior=df[params].copy(),
        parameter_labels=labels,
        parameter_labels_with_unit=labels,
    )
    res.injection_parameters = None
    return res


def add_truth_lines(fig, plot_params, truths):
    if truths is None:
        return fig
    ndim = len(plot_params)
    axes = np.array(fig.axes).reshape((ndim, ndim))
    for i, yparam in enumerate(plot_params):
        ytruth = truths.get(yparam, None)
        if ytruth is not None and np.isfinite(ytruth):
            axes[i, i].axvline(ytruth, color="red", lw=1.5)
        for j in range(i):
            xparam = plot_params[j]
            xtruth = truths.get(xparam, None)
            ax = axes[i, j]
            if xtruth is not None and np.isfinite(xtruth):
                ax.axvline(xtruth, color="red", lw=1.0)
            if ytruth is not None and np.isfinite(ytruth):
                ax.axhline(ytruth, color="red", lw=1.0)
    return fig


def build_clean_transformed_prior(result, params, transform=None, target_size=None, seed=123456, oversample_factor=4, min_batch=4000, max_rounds=8):
    target_size = len(result.posterior) if target_size is None else int(target_size)
    pieces = []
    n_good = 0

    for i in range(max_rounds):
        batch_size = max(min_batch, oversample_factor * target_size)
        df = sample_prior_dataframe(result, size=batch_size, seed=seed + i)
        if transform is not None:
            df = transform(df)
        df = finite_clean(df, params)
        if len(df) > 0:
            pieces.append(df)
            n_good += len(df)
        if n_good >= target_size:
            break

    if n_good == 0:
        raise RuntimeError("No valid prior samples survived after transformation/cleaning.")

    out = pd.concat(pieces, ignore_index=True)
    if len(out) > target_size:
        out = out.sample(n=target_size, random_state=seed).reset_index(drop=True)
    return out


def plot_bilby_multiple(result_list, labels, plot_params, truths=None, bins=40, plot_datapoints=False, smooth=None, smooth1d=None, **kwargs):
    plot_kwargs = dict(
        labels=labels,
        parameters=plot_params,
        save=False,
        bins=bins,
        plot_datapoints=plot_datapoints,
        **kwargs,
    )
    if smooth is not None:
        plot_kwargs["smooth"] = smooth
    if smooth1d is not None:
        plot_kwargs["smooth1d"] = smooth1d

    fig = bilby.core.result.plot_multiple(
        result_list,
        **plot_kwargs,
    )
    add_truth_lines(fig, plot_params, truths)
    plt.show()
    return fig


# =========================================================
# EOS-fit helpers aligned with PE_eosfit_reweight.py
# =========================================================
H0_TRUE = float(Planck18.H0.value)
LAMBDA_FIT_NORM = 3500.0
A0_FIT = -0.30781804
A1_FIT = 0.79244108
A2_FIT = -0.51480556
DELTA_TRUE = 0.0

cosmo_fid = FlatLambdaCDM(H0=70.0, Om0=float(Planck18.Om0), Tcmb0=Planck18.Tcmb0)
_z_grid = np.linspace(0.0, 2.0, 20000)
_dL_grid = np.asarray(cosmo_fid.luminosity_distance(_z_grid).value, dtype=float)


def z_from_dL_H0_vec(dL_mpc, H0):
    dL_scaled = np.asarray(dL_mpc, dtype=float) * (np.asarray(H0, dtype=float) / 70.0)
    return np.interp(np.clip(dL_scaled, _dL_grid[0], _dL_grid[-1]), _dL_grid, _z_grid)


def lambda_of_mbar_vec(mbar, delta_a0, delta_a1, delta_a2):
    mbar = np.asarray(mbar, dtype=float)
    poly = (
        1.0
        + A0_FIT * (1.0 + np.asarray(delta_a0, dtype=float))
        + A1_FIT * (1.0 + np.asarray(delta_a1, dtype=float)) * mbar
        + A2_FIT * (1.0 + np.asarray(delta_a2, dtype=float)) * mbar**2
    )
    lam = LAMBDA_FIT_NORM * poly / mbar**5
    lam = np.where(np.isfinite(lam), lam, 0.0)
    return np.maximum(lam, 1e-8)


def add_eosfit_derived_columns(df, h0_default=H0_TRUE):
    df = ensure_dataframe(df)
    df = rename_legacy_mass_columns(df)
    p = df.copy()

    if "luminosity_distance" not in p.columns or "chirp_mass" not in p.columns or "mass_ratio" not in p.columns:
        raise ValueError("EOS-fit conversion needs luminosity_distance, chirp_mass, and mass_ratio columns.")

    dL = p["luminosity_distance"].to_numpy(dtype=float)
    H0 = p["H0_sample"].to_numpy(dtype=float) if "H0_sample" in p.columns else np.full(len(p), h0_default)
    q = p["mass_ratio"].to_numpy(dtype=float)
    Mc = p["chirp_mass"].to_numpy(dtype=float)

    m1, m2 = bilby.gw.conversion.chirp_mass_and_mass_ratio_to_component_masses(Mc, q)
    z = z_from_dL_H0_vec(dL, H0)
    m1_src = np.asarray(m1, dtype=float) / (1.0 + z)
    m2_src = np.asarray(m2, dtype=float) / (1.0 + z)

    delta_a0 = p["delta_a0"].to_numpy(dtype=float) if "delta_a0" in p.columns else np.zeros(len(p), dtype=float)
    delta_a1 = p["delta_a1"].to_numpy(dtype=float) if "delta_a1" in p.columns else np.zeros(len(p), dtype=float)
    delta_a2 = p["delta_a2"].to_numpy(dtype=float) if "delta_a2" in p.columns else np.zeros(len(p), dtype=float)

    lam1 = lambda_of_mbar_vec(m1_src, delta_a0, delta_a1, delta_a2)
    lam2 = lambda_of_mbar_vec(m2_src, delta_a0, delta_a1, delta_a2)

    p["mass_1"] = m1
    p["mass_2"] = m2
    p["mass_1_source"] = m1_src
    p["mass_2_source"] = m2_src
    p["redshift_sample"] = z
    p["lambda_1"] = lam1
    p["lambda_2"] = lam2
    p["lambda_tilde"] = bilby.gw.conversion.lambda_1_lambda_2_to_lambda_tilde(lam1, lam2, m1, m2)
    p["delta_lambda_tilde"] = bilby.gw.conversion.lambda_1_lambda_2_to_delta_lambda_tilde(lam1, lam2, m1, m2)
    return p


# =========================================================
# Traditional helpers aligned with PE_Traditional.py
# =========================================================
def convert_traditional_bns(parameters):
    is_df = isinstance(parameters, pd.DataFrame)
    if is_df:
        df = rename_legacy_mass_columns(parameters)
        p = {k: df[k].to_numpy() for k in df.columns}
    else:
        p = dict(parameters)

    for idx in ("1", "2"):
        p.pop(f"chi_{idx}_in_plane", None)
        if f"chi_{idx}" in p and f"a_{idx}" in p:
            try:
                if float(p[f"a_{idx}"]) == 0.0:
                    p.pop(f"a_{idx}", None)
                    p.pop(f"cos_tilt_{idx}", None)
                    p.pop(f"tilt_{idx}", None)
            except Exception:
                pass

    converted, _ = bilby.gw.conversion.convert_to_lal_binary_neutron_star_parameters(p)
    converted = bilby.gw.conversion.generate_mass_parameters(converted)
    converted = bilby.gw.conversion.generate_tidal_parameters(converted)

    if is_df:
        return rename_legacy_mass_columns(pd.DataFrame(converted, index=df.index))
    return converted


def ensure_traditional_augmented(df):
    df = ensure_dataframe(df)
    needed = {"mass_1", "mass_2", "lambda_1", "lambda_2"}
    if needed.issubset(df.columns):
        return rename_legacy_mass_columns(df)
    return rename_legacy_mass_columns(ensure_dataframe(convert_traditional_bns(df)))


# =========================================================
# Labels / truths
# =========================================================
LABEL_MAP = {
    "chirp_mass": r"$\mathcal{M}_c$",
    "mass_ratio": r"$q$",
    "luminosity_distance": r"$d_L$",
    "redshift_sample": r"$z$",
    "H0_sample": r"$H_0$",
    "delta_a0": r"$\delta a_0$",
    "delta_a1": r"$\delta a_1$",
    "delta_a2": r"$\delta a_2$",
    "lambda_tilde": r"$\tilde{\Lambda}$",
    "delta_lambda_tilde": r"$\delta\tilde{\Lambda}$",
    "lambda_1": r"$\Lambda_1$",
    "lambda_2": r"$\Lambda_2$",
    "mass_1": r"$m_1^{\rm det}$",
    "mass_2": r"$m_2^{\rm det}$",
}



In [3]:
# =========================================================
# Load metadata and both datasets
# =========================================================
meta = load_meta(POP_OUTDIR, EVENT_NAME)
inj = dict(meta["injection_parameters"])
m1_det = float(meta["mass_1_detector"])
m2_det = float(meta["mass_2_detector"])

z_truth = float(z_from_dL_H0_vec(float(inj["luminosity_distance"]), H0_TRUE))

truth_common = {
    "chirp_mass": (m1_det * m2_det) ** (3.0 / 5.0) / (m1_det + m2_det) ** (1.0 / 5.0),
    "mass_ratio": min(m1_det, m2_det) / max(m1_det, m2_det),
    "luminosity_distance": float(inj["luminosity_distance"]),
    "mass_1": m1_det,
    "mass_2": m2_det,
    "lambda_tilde": bilby.gw.conversion.lambda_1_lambda_2_to_lambda_tilde(
        float(inj["lambda_1"]), float(inj["lambda_2"]), m1_det, m2_det
    ),
    "delta_lambda_tilde": bilby.gw.conversion.lambda_1_lambda_2_to_delta_lambda_tilde(
        float(inj["lambda_1"]), float(inj["lambda_2"]), m1_det, m2_det
    ),
    "redshift_sample": z_truth,
    "H0_sample": float(H0_TRUE),
    "delta_a0": 0.0,
    "delta_a1": 0.0,
    "delta_a2": 0.0,
}

eosfit_loaded = load_posterior_dataset(
    EOSFIT_OUTDIR,
    EOSFIT_LABEL,
    rw_label=EOSFIT_RW_LABEL,
    use_reweighted=USE_REWEIGHTED_EOSFIT,
)
trad_loaded = load_posterior_dataset(
    TRAD_OUTDIR,
    TRAD_LABEL,
    rw_label=TRAD_RW_LABEL,
    use_reweighted=USE_REWEIGHTED_TRAD,
)

# Raw posteriors
post_eosfit_raw = eosfit_loaded["posterior_df_raw"]
post_trad_raw = trad_loaded["posterior_df_raw"]

# Converted / derived posteriors
post_eosfit = add_eosfit_derived_columns(post_eosfit_raw)
post_trad = ensure_traditional_augmented(post_trad_raw)

# Priors are sampled from the original result objects
# If N_PRIOR is None, match the number of rows in the loaded posterior tables.
n_prior_eosfit = len(post_eosfit_raw) if N_PRIOR is None else int(N_PRIOR)
n_prior_trad = len(post_trad_raw) if N_PRIOR is None else int(N_PRIOR)

raw_prior_eosfit = sample_prior_dataframe(
    eosfit_loaded["base_result"], size=n_prior_eosfit, seed=PRIOR_RANDOM_SEED
)
raw_prior_trad = sample_prior_dataframe(
    trad_loaded["base_result"], size=n_prior_trad, seed=PRIOR_RANDOM_SEED + 1
)
prior_eosfit = add_eosfit_derived_columns(raw_prior_eosfit)
prior_eosfit = apply_result_constraints(prior_eosfit, eosfit_loaded["base_result"])

prior_trad = ensure_traditional_augmented(raw_prior_trad)
prior_trad = apply_result_constraints(prior_trad, trad_loaded["base_result"])

print("EOS-fit posterior source:", eosfit_loaded["posterior_source"])
print("Traditional posterior source:", trad_loaded["posterior_source"])
print("EOS-fit base result for prior sampling:", eosfit_loaded["base_result_path"])
print("Traditional base result for prior sampling:", trad_loaded["base_result_path"])
if eosfit_loaded.get("summary_data") is not None:
    print("EOS-fit reweight method:", eosfit_loaded["summary_data"].get("rw_method", eosfit_loaded["summary_data"].get("resampling_method")))
    print("EOS-fit reweight summary:", eosfit_loaded["summary_path"])
if trad_loaded.get("summary_data") is not None:
    print("Traditional reweight method:", trad_loaded["summary_data"].get("rw_method", trad_loaded["summary_data"].get("resampling_method")))
    print("Traditional reweight summary:", trad_loaded["summary_path"])
print()
print("EOS-fit posterior samples:", len(post_eosfit))
print("EOS-fit prior samples    :", len(prior_eosfit))
print("Traditional posterior samples:", len(post_trad))
print("Traditional prior samples    :", len(prior_trad))

print("Traditional posterior type:", type(post_trad).__name__, "shape:", getattr(post_trad, "shape", None))
print("Traditional prior type    :", type(prior_trad).__name__, "shape:", getattr(prior_trad, "shape", None))
print("Raw traditional prior shape:", getattr(raw_prior_trad, "shape", None))

if "lambda_1" in prior_eosfit.columns and "lambda_2" in prior_eosfit.columns:
    print("EOS-fit prior lambda_1 range after constraints:", float(prior_eosfit["lambda_1"].min()), float(prior_eosfit["lambda_1"].max()))
    print("EOS-fit prior lambda_2 range after constraints:", float(prior_eosfit["lambda_2"].min()), float(prior_eosfit["lambda_2"].max()))
if "lambda_tilde" in prior_eosfit.columns:
    print("EOS-fit prior lambda_tilde range:", float(prior_eosfit["lambda_tilde"].min()), float(prior_eosfit["lambda_tilde"].max()))



16:17 bilby INFO    : Global meta data was removed from the result object for compatibility. Use the `BILBY_INCLUDE_GLOBAL_METADATA` environment variable to include it. This behaviour will be removed in a future release. For more details see: https://bilby-dev.github.io/bilby/faq.html#global-meta-data


FileNotFoundError: Cannot find result file. Tried:
  /scratch/gpfs/ANDREASB/fy6204/GW/Workspace/outputs/outdir_population_run_traditional/bns_event_0005_traditional_Mc0.1_result.hdf5
  /scratch/gpfs/ANDREASB/fy6204/GW/Workspace/outputs/outdir_population_run_traditional/bns_event_0005_traditional_Mc0.1_result.h5
  /scratch/gpfs/ANDREASB/fy6204/GW/Workspace/outputs/outdir_population_run_traditional/bns_event_0005_traditional_Mc0.1_result.json
  /scratch/gpfs/ANDREASB/fy6204/GW/Workspace/outputs/outdir_population_run_traditional/bns_event_0005_traditional_Mc0.1_result.pkl

In [ ]:
# =========================================================
# 1) EOS-fit / UR parameterization: prior vs posterior
#    Two versions: before reweight and after reweight
# =========================================================
def prepare_eosfit_prior_posterior(use_reweighted, seed_offset=0):
    loaded = load_posterior_dataset(
        EOSFIT_OUTDIR,
        EOSFIT_LABEL,
        rw_label=EOSFIT_RW_LABEL,
        use_reweighted=use_reweighted,
    )

    post_raw = loaded["posterior_df_raw"]
    post_df_full = add_eosfit_derived_columns(post_raw)

    n_prior = len(post_raw) if N_PRIOR is None else int(N_PRIOR)
    raw_prior = sample_prior_dataframe(
        loaded["base_result"],
        size=n_prior,
        seed=PRIOR_RANDOM_SEED + seed_offset,
    )
    prior_df_full = add_eosfit_derived_columns(raw_prior)
    prior_df_full = apply_result_constraints(prior_df_full, loaded["base_result"])

    return loaded, post_df_full, prior_df_full


def plot_eosfit_ur_prior_posterior(use_reweighted, tag, seed_offset=0):
    loaded, post_df_full, prior_df_full = prepare_eosfit_prior_posterior(use_reweighted, seed_offset=seed_offset)
    params = [p for p in EOSFIT_UR_PARAMS if p in prior_df_full.columns and p in post_df_full.columns]

    prior_df = finite_clean(prior_df_full, params)
    post_df = finite_clean(post_df_full, params)
    post_df = maybe_cap_reweighted_duplicates(
        post_df,
        keys=params,
        enabled=bool(use_reweighted and CAP_REWEIGHTED_DUPLICATES_FOR_PLOT),
        max_per_point=int(MAX_DUPLICATES_PER_POINT),
        round_decimals=int(DUPLICATE_ROUND_DECIMALS),
        random_state=int(DUPLICATE_CAP_RANDOM_SEED + seed_offset),
        tag=f"EOS UR {tag}",
    )

    priors_for_plot = build_priors_for_plot(loaded["base_result"], post_df, params)
    prior_result = make_lightweight_result(
        loaded["base_result"],
        prior_df,
        params,
        LABEL_MAP,
        result_label=f"{EOSFIT_LABEL}_ur_prior_overlay_{tag}",
        sampler="prior",
        priors_for_plot=priors_for_plot,
    )
    post_result = make_lightweight_result(
        loaded["base_result"],
        post_df,
        params,
        LABEL_MAP,
        result_label=f"{loaded['active_label']}_ur_posterior_overlay_{tag}",
        sampler=loaded["base_result"].sampler,
        priors_for_plot=priors_for_plot,
    )

    plot_bilby_multiple(
        [post_result, prior_result],
        labels=[f"posterior ({tag})", f"prior ({tag})"],
        plot_params=params,
        truths=truth_common,
        bins=40,
    )

    print(f"[EOS UR {tag}] plotted parameters:", params)
    print(f"[EOS UR {tag}] posterior samples:", len(post_df), "prior samples:", len(prior_df))


plot_eosfit_ur_prior_posterior(False, "before RW", seed_offset=0)
plot_eosfit_ur_prior_posterior(True,  "after RW",  seed_offset=100)



In [4]:
# =========================================================
# 2) EOS-fit mapped to common physical variables: prior vs posterior
#    Two versions: before reweight and after reweight
# =========================================================
def plot_eosfit_physical_prior_posterior(use_reweighted, tag, seed_offset=0):
    loaded, post_df_full, prior_df_full = prepare_eosfit_prior_posterior(use_reweighted, seed_offset=seed_offset)
    params = [p for p in COMMON_PHYSICAL_PARAMS if p in prior_df_full.columns and p in post_df_full.columns]

    prior_df = finite_clean(prior_df_full, params)
    post_df = finite_clean(post_df_full, params)
    post_df = maybe_cap_reweighted_duplicates(
        post_df,
        keys=params,
        enabled=bool(use_reweighted and CAP_REWEIGHTED_DUPLICATES_FOR_PLOT),
        max_per_point=int(MAX_DUPLICATES_PER_POINT),
        round_decimals=int(DUPLICATE_ROUND_DECIMALS),
        random_state=int(DUPLICATE_CAP_RANDOM_SEED + seed_offset),
        tag=f"EOS physical {tag}",
    )

    priors_for_plot = build_priors_for_plot(loaded["base_result"], post_df, params)
    prior_result = make_lightweight_result(
        loaded["base_result"],
        prior_df,
        params,
        LABEL_MAP,
        result_label=f"{EOSFIT_LABEL}_physical_prior_overlay_{tag}",
        sampler="prior",
        priors_for_plot=priors_for_plot,
    )
    post_result = make_lightweight_result(
        loaded["base_result"],
        post_df,
        params,
        LABEL_MAP,
        result_label=f"{loaded['active_label']}_physical_posterior_overlay_{tag}",
        sampler=loaded["base_result"].sampler,
        priors_for_plot=priors_for_plot,
    )

    plot_bilby_multiple(
        [post_result, prior_result],
        labels=[f"posterior ({tag})", f"prior ({tag})"],
        plot_params=params,
        truths=truth_common,
        bins=40,
    )

    print(f"[EOS physical {tag}] plotted parameters:", params)
    print(f"[EOS physical {tag}] posterior samples:", len(post_df), "prior samples:", len(prior_df))


plot_eosfit_physical_prior_posterior(False, "before RW", seed_offset=10)
plot_eosfit_physical_prior_posterior(True,  "after RW",  seed_offset=110)



NameError: name 'prepare_eosfit_prior_posterior' is not defined

In [ ]:
# =========================================================
# 3) Traditional fit in common physical variables: prior vs posterior
#    Two versions: before reweight and after reweight
# =========================================================
def prepare_trad_prior_posterior(use_reweighted, seed_offset=0):
    loaded = load_posterior_dataset(
        TRAD_OUTDIR,
        TRAD_LABEL,
        rw_label=TRAD_RW_LABEL,
        use_reweighted=use_reweighted,
    )

    post_raw = loaded["posterior_df_raw"]
    post_df_full = ensure_traditional_augmented(post_raw)

    n_prior = len(post_raw) if N_PRIOR is None else int(N_PRIOR)
    raw_prior = sample_prior_dataframe(
        loaded["base_result"],
        size=n_prior,
        seed=PRIOR_RANDOM_SEED + 1 + seed_offset,
    )
    prior_df_full = ensure_traditional_augmented(raw_prior)
    prior_df_full = apply_result_constraints(prior_df_full, loaded["base_result"])

    return loaded, post_df_full, prior_df_full


def plot_trad_physical_prior_posterior(use_reweighted, tag, seed_offset=0):
    loaded, post_df_full, prior_df_full = prepare_trad_prior_posterior(use_reweighted, seed_offset=seed_offset)
    params = [p for p in COMMON_PHYSICAL_PARAMS if p in prior_df_full.columns and p in post_df_full.columns]

    prior_df = finite_clean(prior_df_full, params)
    post_df = finite_clean(post_df_full, params)
    post_df = maybe_cap_reweighted_duplicates(
        post_df,
        keys=params,
        enabled=bool(use_reweighted and CAP_REWEIGHTED_DUPLICATES_FOR_PLOT),
        max_per_point=int(MAX_DUPLICATES_PER_POINT),
        round_decimals=int(DUPLICATE_ROUND_DECIMALS),
        random_state=int(DUPLICATE_CAP_RANDOM_SEED + seed_offset),
        tag=f"Traditional physical {tag}",
    )

    priors_for_plot = build_priors_for_plot(loaded["base_result"], post_df, params)
    prior_result = make_lightweight_result(
        loaded["base_result"],
        prior_df,
        params,
        LABEL_MAP,
        result_label=f"{TRAD_LABEL}_physical_prior_overlay_{tag}",
        sampler="prior",
        priors_for_plot=priors_for_plot,
    )
    post_result = make_lightweight_result(
        loaded["base_result"],
        post_df,
        params,
        LABEL_MAP,
        result_label=f"{loaded['active_label']}_physical_posterior_overlay_{tag}",
        sampler=loaded["base_result"].sampler,
        priors_for_plot=priors_for_plot,
    )

    plot_bilby_multiple(
        [post_result, prior_result],
        labels=[f"posterior ({tag})", f"prior ({tag})"],
        plot_params=params,
        truths=truth_common,
        bins=40,
    )

    print(f"[Traditional physical {tag}] plotted parameters:", params)
    print(f"[Traditional physical {tag}] posterior samples:", len(post_df), "prior samples:", len(prior_df))


plot_trad_physical_prior_posterior(False, "before RW", seed_offset=20)
plot_trad_physical_prior_posterior(True,  "after RW",  seed_offset=120)



In [ ]:
# =========================================================
# 4) EOS-fit vs Traditional: posterior comparison in common physical variables
#    Two versions: before reweight and after reweight
# =========================================================
def build_overlay_results(use_reweighted_eosfit, use_reweighted_trad):
    eos_loaded = load_posterior_dataset(
        EOSFIT_OUTDIR,
        EOSFIT_LABEL,
        rw_label=EOSFIT_RW_LABEL,
        use_reweighted=use_reweighted_eosfit,
    )
    tr_loaded = load_posterior_dataset(
        TRAD_OUTDIR,
        TRAD_LABEL,
        rw_label=TRAD_RW_LABEL,
        use_reweighted=use_reweighted_trad,
    )

    eos_df_full = add_eosfit_derived_columns(eos_loaded["posterior_df_raw"])
    tr_df_full = ensure_traditional_augmented(tr_loaded["posterior_df_raw"])
    params = [
        p for p in COMMON_PHYSICAL_PARAMS
        if p in eos_df_full.columns and p in tr_df_full.columns
    ]
    eos_df = finite_clean(eos_df_full, params)
    tr_df = finite_clean(tr_df_full, params)

    eos_df = maybe_cap_reweighted_duplicates(
        eos_df,
        keys=params,
        enabled=bool(use_reweighted_eosfit and CAP_REWEIGHTED_DUPLICATES_FOR_PLOT),
        max_per_point=int(MAX_DUPLICATES_PER_POINT),
        round_decimals=int(DUPLICATE_ROUND_DECIMALS),
        random_state=int(DUPLICATE_CAP_RANDOM_SEED + 700),
        tag="EOS overlay",
    )
    tr_df = maybe_cap_reweighted_duplicates(
        tr_df,
        keys=params,
        enabled=bool(use_reweighted_trad and CAP_REWEIGHTED_DUPLICATES_FOR_PLOT),
        max_per_point=int(MAX_DUPLICATES_PER_POINT),
        round_decimals=int(DUPLICATE_ROUND_DECIMALS),
        random_state=int(DUPLICATE_CAP_RANDOM_SEED + 701),
        tag="Traditional overlay",
    )

    eos_priors_for_plot = build_priors_for_plot(eos_loaded["base_result"], eos_df, params)
    trad_priors_for_plot = build_priors_for_plot(tr_loaded["base_result"], tr_df, params)

    eos_result = make_lightweight_result(
        eos_loaded["base_result"],
        eos_df,
        params,
        LABEL_MAP,
        result_label=eos_loaded["active_label"] + "_vs_traditional_overlay",
        sampler=eos_loaded["base_result"].sampler,
        priors_for_plot=eos_priors_for_plot,
    )
    trad_result = make_lightweight_result(
        tr_loaded["base_result"],
        tr_df,
        params,
        LABEL_MAP,
        result_label=tr_loaded["active_label"] + "_vs_eosfit_overlay",
        sampler=tr_loaded["base_result"].sampler,
        priors_for_plot=trad_priors_for_plot,
    )
    return [eos_result, trad_result], params


# Before reweight
results_before, params_before = build_overlay_results(False, False)
plot_bilby_multiple(
    results_before,
    labels=["EOS-fit posterior (before RW)", "Traditional posterior (before RW)"],
    plot_params=params_before,
    truths=truth_common,
    bins=40,
)
print("Plotted BEFORE reweight parameters:", params_before)

# After reweight
results_after, params_after = build_overlay_results(True, True)
plot_bilby_multiple(
    results_after,
    labels=["EOS-fit posterior (after RW)", "Traditional posterior (after RW)"],
    plot_params=params_after,
    truths=truth_common,
    bins=40,
)
print("Plotted AFTER reweight parameters:", params_after)


In [1]:

# =========================================================
# 5) Reweight diagnostics + before/after posterior overlay
# =========================================================
def _reweight_cache_path(outdir, rw_label):
    return os.path.join(outdir, f"{rw_label}_reweight_arrays.npz")

def load_reweight_cache(cache_path):
    data = np.load(cache_path, allow_pickle=True)
    posterior = pd.DataFrame(data["posterior_dict"][0])
    lnw = np.asarray(data["ln_weights"], dtype=float)
    new_ll = np.asarray(data["new_log_likelihood"], dtype=float)
    new_lp = np.asarray(data["new_log_prior"], dtype=float)
    old_ll = np.asarray(data["old_log_likelihood"], dtype=float)
    old_lp = np.asarray(data["old_log_prior"], dtype=float)
    metadata = dict(data["metadata"][0]) if "metadata" in data else {}
    return posterior, lnw, new_ll, new_lp, old_ll, old_lp, metadata

def normalized_weights_from_lnw(lnw):
    lnw = np.asarray(lnw, dtype=float)
    finite = np.isfinite(lnw)
    if not np.any(finite):
        raise ValueError("No finite log-weights in cache.")
    lnw_f = lnw[finite]
    lnw_shift = lnw_f - np.max(lnw_f)
    w_rel = np.exp(lnw_shift)
    w_norm = w_rel / np.sum(w_rel)
    full = np.zeros_like(lnw, dtype=float)
    full[finite] = w_norm
    return full, finite

def summarize_weights(lnw):
    w, finite = normalized_weights_from_lnw(lnw)
    wf = w[finite]
    ess = 1.0 / np.sum(wf**2)
    return dict(
        n_total=int(len(lnw)),
        n_finite=int(np.sum(finite)),
        ess=float(ess),
        ess_fraction=float(ess / len(wf)),
        max_weight=float(np.max(wf)),
        min_logw_shift=float(np.min(np.asarray(lnw)[finite] - np.max(np.asarray(lnw)[finite]))),
    )

def weighted_quantile(values, quantiles, sample_weight=None):
    values = np.asarray(values, dtype=float)
    quantiles = np.atleast_1d(quantiles)
    if sample_weight is None:
        return np.quantile(values, quantiles)
    w = np.asarray(sample_weight, dtype=float)
    mask = np.isfinite(values) & np.isfinite(w) & (w > 0)
    v = values[mask]
    w = w[mask]
    if len(v) == 0:
        return np.full_like(quantiles, np.nan, dtype=float)
    order = np.argsort(v)
    v = v[order]
    w = w[order]
    cdf = np.cumsum(w)
    cdf = cdf / cdf[-1]
    return np.interp(quantiles, cdf, v)

def plot_single_reweight_diagnostics(outdir, label, rw_label, title_prefix, physical_transform):
    cache_path = _reweight_cache_path(outdir, rw_label)
    if not os.path.exists(cache_path):
        print(f"[{title_prefix}] cache not found: {cache_path}")
        return None

    posterior_raw, lnw, new_ll, new_lp, old_ll, old_lp, metadata = load_reweight_cache(cache_path)
    weights, finite = normalized_weights_from_lnw(lnw)
    diag = summarize_weights(lnw)

    df = physical_transform(posterior_raw)
    df = ensure_dataframe(df)

    delta_logl = np.asarray(new_ll, dtype=float) - np.asarray(old_ll, dtype=float)
    delta_logp = np.asarray(new_lp, dtype=float) - np.asarray(old_lp, dtype=float)

    print(f"[{title_prefix}] cache: {cache_path}")
    print(f"[{title_prefix}] metadata keys:", sorted(metadata.keys()))
    print(f"[{title_prefix}] diagnostics:", json.dumps(diag, indent=2))

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))

    # 1) shifted log-weight histogram
    lnw_f = np.asarray(lnw, dtype=float)[finite]
    axes[0, 0].hist(lnw_f - np.max(lnw_f), bins=60)
    axes[0, 0].set_xlabel(r"$\ln w - \max(\ln w)$")
    axes[0, 0].set_ylabel("count")
    axes[0, 0].set_title(f"{title_prefix}: log-weight spread")

    # 2) sorted normalized weights
    w_sorted = np.sort(weights[finite])[::-1]
    axes[0, 1].plot(np.arange(1, len(w_sorted) + 1), np.cumsum(w_sorted))
    axes[0, 1].set_xlabel("top-ranked samples")
    axes[0, 1].set_ylabel("cumulative normalized weight")
    axes[0, 1].set_title(f"{title_prefix}: weight concentration")

    # 3) delta logL vs lambda_tilde
    if "lambda_tilde" in df.columns:
        axes[1, 0].scatter(df["lambda_tilde"], delta_logl, s=5, alpha=0.25)
        axes[1, 0].set_xlabel(r"$\tilde{\Lambda}$")
        axes[1, 0].set_ylabel(r"$\Delta \log \mathcal{L}_{\rm full-RB}$")
        axes[1, 0].set_title(f"{title_prefix}: likelihood correction")
    else:
        axes[1, 0].text(0.5, 0.5, "lambda_tilde not available", ha="center", va="center")
        axes[1, 0].set_axis_off()

    # 4) weighted vs unweighted 1D hist on a key parameter
    key = "lambda_tilde" if "lambda_tilde" in df.columns else (
        "luminosity_distance" if "luminosity_distance" in df.columns else None
    )
    if key is not None:
        x = pd.to_numeric(df[key], errors="coerce").to_numpy(dtype=float)
        mask = np.isfinite(x) & finite
        axes[1, 1].hist(x[mask], bins=50, density=True, histtype="step", label="before RW")
        axes[1, 1].hist(x[mask], bins=50, density=True, histtype="step", weights=weights[mask], label="weighted")
        q0 = weighted_quantile(x[mask], [0.5], None)[0]
        q1 = weighted_quantile(x[mask], [0.5], weights[mask])[0]
        axes[1, 1].axvline(q0, lw=1, ls="--", label=f"before median={q0:.3g}")
        axes[1, 1].axvline(q1, lw=1, ls=":", label=f"weighted median={q1:.3g}")
        axes[1, 1].set_xlabel(key)
        axes[1, 1].set_ylabel("density")
        axes[1, 1].set_title(f"{title_prefix}: weighted vs unweighted")
        axes[1, 1].legend(fontsize=8)
    else:
        axes[1, 1].text(0.5, 0.5, "no scalar parameter available", ha="center", va="center")
        axes[1, 1].set_axis_off()

    fig.suptitle(f"{title_prefix} reweight diagnostics", y=1.02)
    plt.tight_layout()
    plt.show()

    # quick numeric summary on the most important columns
    for key in ["lambda_tilde", "delta_lambda_tilde", "luminosity_distance", "mass_ratio"]:
        if key in df.columns:
            x = pd.to_numeric(df[key], errors="coerce").to_numpy(dtype=float)
            mask = np.isfinite(x) & finite
            q_before = weighted_quantile(x[mask], [0.16, 0.5, 0.84], None)
            q_after = weighted_quantile(x[mask], [0.16, 0.5, 0.84], weights[mask])
            print(
                f"[{title_prefix}] {key}: "
                f"before=({q_before[0]:.4g}, {q_before[1]:.4g}, {q_before[2]:.4g}) | "
                f"weighted=({q_after[0]:.4g}, {q_after[1]:.4g}, {q_after[2]:.4g})"
            )

    print(f"[{title_prefix}] max ΔlogL = {np.nanmax(delta_logl):.4f}, min ΔlogL = {np.nanmin(delta_logl):.4f}")
    print(f"[{title_prefix}] unique finite weights = {len(np.unique(np.round(weights[finite], 18)))}")

    return dict(df=df, weights=weights, lnw=lnw, new_ll=new_ll, old_ll=old_ll, metadata=metadata, diag=diag)

def overlay_before_after_single_model(outdir, label, rw_label, title_prefix, transform_fn, params=None, seed_offset=0):
    loaded_before = load_posterior_dataset(outdir, label, rw_label=rw_label, use_reweighted=False)
    loaded_after  = load_posterior_dataset(outdir, label, rw_label=rw_label, use_reweighted=True)

    before_full = transform_fn(loaded_before["posterior_df_raw"])
    after_full  = transform_fn(loaded_after["posterior_df_raw"])

    if params is None:
        params = [p for p in COMMON_PHYSICAL_PARAMS if p in before_full.columns and p in after_full.columns]

    before_df = finite_clean(before_full, params)
    after_df = finite_clean(after_full, params)
    after_df = maybe_cap_reweighted_duplicates(
        after_df,
        keys=params,
        enabled=CAP_REWEIGHTED_DUPLICATES_FOR_PLOT,
        max_per_point=MAX_DUPLICATES_PER_POINT,
        round_decimals=DUPLICATE_ROUND_DECIMALS,
        random_state=DUPLICATE_CAP_RANDOM_SEED + seed_offset,
        tag=f"{title_prefix} after-RW overlay",
    )

    combined_for_priors = pd.concat([before_df[params], after_df[params]], ignore_index=True)
    priors_for_plot = build_priors_for_plot(loaded_before["base_result"], combined_for_priors, params)

    before_result = make_lightweight_result(
        loaded_before["base_result"],
        before_df,
        params,
        LABEL_MAP,
        result_label=f"{label}_before_rw_overlay",
        sampler=loaded_before["base_result"].sampler,
        priors_for_plot=priors_for_plot,
    )
    after_result = make_lightweight_result(
        loaded_after["base_result"],
        after_df,
        params,
        LABEL_MAP,
        result_label=f"{rw_label}_after_rw_overlay",
        sampler=loaded_after["base_result"].sampler,
        priors_for_plot=priors_for_plot,
    )

    fig = plot_bilby_multiple(
        [before_result, after_result],
        labels=[f"{title_prefix} before RW", f"{title_prefix} after RW"],
        plot_params=params,
        truths=truth_common,
        bins=40,
    )
    print(f"[{title_prefix}] overlaid before/after RW parameters:", params)
    print(f"[{title_prefix}] before samples={len(before_df)}, after samples={len(after_df)}")
    return fig, params

# --- diagnostics from cache ---
_ = plot_single_reweight_diagnostics(
    EOSFIT_OUTDIR,
    EOSFIT_LABEL,
    EOSFIT_RW_LABEL,
    title_prefix="EOS-fit",
    physical_transform=add_eosfit_derived_columns,
)

# Try traditional diagnostics only if a cache exists for it.
trad_cache_path = _reweight_cache_path(TRAD_OUTDIR, TRAD_RW_LABEL)
if os.path.exists(trad_cache_path):
    _ = plot_single_reweight_diagnostics(
        TRAD_OUTDIR,
        TRAD_LABEL,
        TRAD_RW_LABEL,
        title_prefix="Traditional",
        physical_transform=ensure_traditional_augmented,
    )
else:
    print(f"[Traditional] cache not found, skip diagnostics: {trad_cache_path}")

# --- before/after overlay corner plots ---
overlay_before_after_single_model(
    EOSFIT_OUTDIR,
    EOSFIT_LABEL,
    EOSFIT_RW_LABEL,
    title_prefix="EOS-fit",
    transform_fn=add_eosfit_derived_columns,
    params=COMMON_PHYSICAL_PARAMS,
    seed_offset=300,
)

overlay_before_after_single_model(
    TRAD_OUTDIR,
    TRAD_LABEL,
    TRAD_RW_LABEL,
    title_prefix="Traditional",
    transform_fn=ensure_traditional_augmented,
    params=COMMON_PHYSICAL_PARAMS,
    seed_offset=400,
)


NameError: name 'EOSFIT_OUTDIR' is not defined